**© Copyright AIDENTIFY. All rights reserved.**

# Part 4 | Session 03: 벡터 DB 심층 비교 분석 및 실습

## 학습 목표

1. 벡터 DB의 핵심 개념과 전통 DB와의 차이를 이해한다
2. **벡터 DB의 "성능"을 구성하는 5개 축(Recall, 지연, 처리량, 색인 구축, 메모리)을 정의할 수 있다**
3. 주요 벡터 DB(ChromaDB, FAISS, Pinecone, Weaviate, Milvus)의 특징과 장단점을 비교한다
4. **ChromaDB와 FAISS를 동일한 조건(같은 데이터·쿼리·척도)에서 직접 측정해 비교한다**
5. **인덱스 알고리즘(Flat / IVF / HNSW)의 트레이드오프를 수치로 확인한다**
6. 용도별 벡터 DB 선택 기준을 수립한다

---

### 이 노트북의 구성

```
개념      1️⃣ 벡터 DB란        2️⃣ 주요 DB 비교표
  │
정의      3️⃣ "성능"이란 무엇인가  ← 무엇을 잴 것인지 먼저 정한다
  │           (5개 축 + 공정 비교 3원칙)
  │
측정      4️⃣ 공통 데이터 → 5️⃣ 구현체별 사용법 → 6️⃣ 동일 조건 벤치마크
  │                                        (DB를 바꿔가며 측정)
  │       7️⃣ 인덱스 알고리즘 벤치마크
  │           (규모를 키워 인덱스를 바꿔가며 측정)
결론      8️⃣ 선택 가이드
```

3️⃣에서 정의한 지표가 6️⃣·7️⃣ 표의 각 열이 됩니다.

### 실습 환경
- **GPU**: 불필요 (CPU로 충분)
- **필수 패키지**: chromadb, faiss-cpu, sentence-transformers, numpy
- **예상 소요**: 전체 실행 약 1분 (7️⃣ 대규모 벤치마크가 약 30초, 임베딩 모델 최초 다운로드 시 추가)


In [ ]:
# 💡 setup.sh 실행했으면 이 셀은 건너뛰세요 (참고용 — 본 노트북이 필요로 하는 패키지)
# 필수 패키지 설치
# !pip install -q chromadb faiss-cpu sentence-transformers numpy

In [ ]:
# 패키지 버전 확인
import importlib

packages = [
    "chromadb",
    "faiss",
    "sentence_transformers",
    "numpy",
]

print("패키지 버전 확인")
print("=" * 40)

for pkg_name in packages:
    try:
        pkg = importlib.import_module(pkg_name)
        version = getattr(pkg, "__version__", "installed")
        print(f"  [OK] {pkg_name}: {version}")
    except ImportError:
        print(f"  [X]  {pkg_name}: 설치 필요")

---

## 1️⃣ 벡터 DB란? 전통 DB vs 벡터 DB 차이

### 전통 데이터베이스 (RDBMS)
- **저장 방식**: 행(row)과 열(column)로 구조화된 데이터
- **검색 방식**: SQL 쿼리, 정확한 키워드 매칭 (WHERE, LIKE)
- **적합한 작업**: 정형 데이터 관리, 트랜잭션 처리

### 벡터 데이터베이스
- **저장 방식**: 고차원 벡터(임베딩)로 변환된 비정형 데이터
- **검색 방식**: 유사도 검색 (코사인 유사도, 유클리드 거리, 내적)
- **적합한 작업**: 시맨틱 검색, 추천 시스템, RAG, 이미지 검색

### 비교표

| 구분 | 전통 DB (RDBMS) | 벡터 DB |
|------|-----------------|----------|
| **데이터 형태** | 정형 데이터 (숫자, 문자열) | 고차원 벡터 (임베딩) |
| **검색 방식** | 정확 매칭 (=, LIKE) | 유사도 기반 (ANN) |
| **인덱싱** | B-Tree, Hash | HNSW, IVF, PQ |
| **결과** | 정확히 일치하는 행 | 가장 유사한 K개 |
| **쿼리 예시** | `SELECT * WHERE name='AI'` | `query(vector, top_k=5)` |
| **주요 사례** | 금융, ERP, CRM | RAG, 추천, 이미지 검색 |

### 벡터 검색의 핵심: ANN (Approximate Nearest Neighbor)

```
  정확 검색 (Brute Force)         근사 검색 (ANN)
  ┌─────────────────┐          ┌─────────────────┐
  │ 모든 벡터와 비교   │          │ 인덱스로 후보 축소  │
  │ O(N) 시간 복잡도  │          │ O(log N) 수준     │
  │ 100% 정확        │          │ 99%+ 정확 (충분)   │
  │ 대규모 시 느림     │          │ 대규모에서도 빠름   │
  └─────────────────┘          └─────────────────┘
```

여기서 **"99%+ 정확"이 정확히 무슨 뜻인지**가 3️⃣의 주제입니다.
근사 검색은 정답을 일부 놓치는 대신 속도를 얻는데, 얼마나 놓치고 얼마나 빨라지는지를
숫자로 말할 수 없으면 벡터 DB를 고를 수 없습니다.


---

## 2️⃣ 주요 벡터 DB 비교: ChromaDB, FAISS, Pinecone, Weaviate, Milvus

### 주요 벡터 DB 종합 비교표

| 항목 | ChromaDB | FAISS | Pinecone | Weaviate | Milvus | pgvector |
|------|----------|-------|----------|----------|--------|----------|
| **개발사** | Chroma | Meta | Pinecone | Weaviate | Zilliz | PostgreSQL 확장 |
| **라이선스** | Apache 2.0 | MIT | 상용 (무료 티어) | BSD-3 | Apache 2.0 | PostgreSQL |
| **배포 방식** | 임베디드/서버 | 라이브러리 | 클라우드 관리형 | 자체 호스팅/클라우드 | 자체 호스팅/클라우드 | PostgreSQL 확장 |
| **언어** | Python | C++/Python | REST API | Go/Python | Go/Python | C/SQL |
| **인덱스** | HNSW | IVF, PQ, HNSW 등 | 독자 알고리즘 | HNSW | IVF, HNSW, DiskANN | IVFFlat, HNSW |
| **확장성** | 소~중규모 | 대규모 (GPU 지원) | 대규모 | 대규모 | 초대규모 (10억+) | 중규모 |
| **메타데이터 필터** | O | X (직접 구현) | O | O | O | O (SQL) |
| **영속성** | O | 수동 저장 | 클라우드 자동 | O | O | O (PostgreSQL) |
| **난이도** | 매우 쉬움 | 중간 | 쉬움 | 중간 | 중~상 | SQL 필요 |

> 이 표의 **"확장성"과 "난이도"는 벤더 문서와 커뮤니티 통념을 정리한 정성 평가**입니다.
> 실측이 아닙니다. 3️⃣ 이후에서 ChromaDB와 FAISS만 실제로 측정해 이 표의 주장을 검증합니다.
> 나머지는 클라우드 계정이나 별도 서버가 필요해 이 노트북에서는 다루지 않습니다.

### 각 벡터 DB의 특징

**ChromaDB**
- 장점: API가 매우 직관적, Python 네이티브, 메타데이터 필터링 내장, 빠른 프로토타이핑
- 단점: 대규모 프로덕션에는 부족, GPU 미지원
- 추천: 학습, 프로토타입, 소규모 RAG 프로젝트

**FAISS (Facebook AI Similarity Search)**
- 장점: 초고속 검색, GPU 가속 지원, 다양한 인덱스 알고리즘, 메모리 효율적
- 단점: 메타데이터 필터링 없음, DB 기능 부재 (라이브러리), CRUD 제한적
- 추천: 대규모 벡터 검색, 성능이 최우선인 경우

**Pinecone**
- 장점: 완전 관리형 서비스, 설정 불필요, 자동 스케일링, 높은 가용성
- 단점: 유료(대규모 시), 벤더 종속, 데이터가 외부 클라우드에 저장
- 추천: 빠른 프로덕션 배포, 인프라 관리 부담을 줄이고 싶은 경우

**Weaviate**
- 장점: GraphQL 지원, 내장 벡터화 모듈, 하이브리드 검색 (키워드+벡터)
- 단점: 리소스 사용량 높음, 학습 곡선
- 추천: 복잡한 스키마가 필요한 경우, 하이브리드 검색

**Milvus**
- 장점: 10억 벡터 이상 처리 가능, 분산 아키텍처, 다양한 인덱스
- 단점: 설치/운영 복잡, 리소스 요구량 높음
- 추천: 초대규모 벡터 검색, 엔터프라이즈 환경

**PostgreSQL + pgvector**
- 장점: 기존 PostgreSQL 인프라 활용, SQL로 벡터 검색, 정형+벡터 데이터 통합
- 단점: 전용 벡터 DB 대비 성능 제한, 대규모 시 속도 저하
- 추천: 이미 PostgreSQL을 사용 중인 경우, 정형 데이터와 벡터를 함께 관리


---

## 3️⃣ 벡터 DB의 "성능"이란 무엇인가

> **"어느 벡터 DB가 제일 빠른가요?"는 대답할 수 없는 질문입니다.**
> 무엇을 재는지 정하지 않았기 때문입니다.

벡터 검색의 성능은 하나의 숫자가 아니라, **서로 맞바꿔야 하는 5개의 축**입니다.

### 성능 5축

| 축 | 지표 | 무엇을 재나 | 나빠지면 생기는 일 |
|---|---|---|---|
| **① 검색 품질** | `Recall@k` | 완전탐색 정답 k개 중 몇 개를 찾았나 | RAG가 근거 문서를 놓쳐 환각 발생 |
| **② 검색 지연** | `p50` / `p95` (ms) | 쿼리 1건의 응답 시간 | 사용자가 느리다고 체감 |
| **③ 처리량** | `QPS` | 초당 처리 가능한 쿼리 수 | 동시 접속자를 못 받음 |
| **④ 색인 구축** | `build time` (s) | 인덱스를 만드는 데 걸리는 시간 | 문서 갱신 주기가 길어짐 |
| **⑤ 메모리** | `index size` (MB) | 인덱스가 차지하는 RAM | 서버 비용 상승, 장비 한계 |

여기에 숫자로 재지 않지만 실무에서 결정적인 **⑥ 운영 특성**이 붙습니다 —
메타데이터 필터링, 원문 저장, 영속성, 실시간 추가/삭제, 인프라 부담.

### ① Recall@k — "정확도"를 정의하는 법

근사 검색이 얼마나 정답을 놓쳤는지는 **완전탐색(Brute Force) 결과를 정답으로 두고** 겹치는 개수로 잽니다.

```
완전탐색 top-5 (정답) : [12,  7, 40,  3, 91]
근사검색  top-5       : [12, 40,  3, 55, 91]
                         ✓       ✓   ✓  ✗   ✓
겹치는 것 = {12, 40, 3, 91} = 4개
Recall@5 = 4 / 5 = 80%
```

**주의**: Recall은 "순서"가 아니라 "집합"을 봅니다. 순서가 뒤바뀌어도 같은 문서가 들어 있으면 정답 처리입니다.
RAG에서는 검색된 문서 전체가 LLM 컨텍스트로 들어가므로, 대개 순서보다 집합이 중요합니다.

### ② 왜 평균이 아니라 p95인가

**평균 지연시간은 느린 쿼리를 감춥니다.**

```
100건의 응답시간: 95건이 10ms, 5건이 2000ms
  평균 = 109ms   → "괜찮은데?"
  p95  = 2000ms  → 20명 중 1명은 2초를 기다린다
```

실서비스 SLA는 항상 p95 / p99로 씁니다. 이 노트북도 p50(중앙값)과 p95를 함께 출력합니다.

### 핵심 트레이드오프: 셋 다 가질 수 없다

```
   정확도(Recall)
     100% ┤ ● FlatL2
          │   (완전탐색: 정확하지만 느리고 메모리 큼)
          │
      90% ┤        ● HNSW          ● IVF(nprobe 높음)
          │          (빠름, 메모리 큼,   (정확도 조절 가능,
          │           구축 느림)          구축 빠름)
      50% ┤                  ● IVF(nprobe 낮음)
          │                    (매우 빠름, 부정확)
          └────────────────────────────────▶ 속도(QPS)
```

**"어느 축을 포기할 것인가"를 정하는 일이 벡터 DB 설계의 전부**입니다.
7️⃣에서 이 그림을 실제 숫자로 그립니다.

### 공정한 비교의 3원칙

이후 모든 실습은 아래를 지킵니다. **하나라도 어기면 비교가 아니라 착시입니다.**

1. **같은 데이터** — 동일한 문서 10개, 동일한 임베딩 모델
2. **같은 쿼리** — 동일한 질의 5개
3. **같은 거리 척도** — 전부 코사인 유사도

3번이 가장 자주 깨집니다. ChromaDB는 코사인이 기본인데 FAISS `IndexFlatL2`는 L2 거리를 씁니다.
그대로 비교하면 한쪽은 `유사도 0.74`, 다른 쪽은 `L2거리 0.51` — **단위가 다른 숫자**가 나옵니다.
순위가 달라져도 그게 DB 성능 차이인지 척도 차이인지 구분할 수 없습니다.

**해결책은 벡터 정규화입니다.** 길이가 1인 벡터 a, b에 대해

```
코사인유사도(a, b) = a · b                 ← 내적과 완전히 같음
L2거리²(a, b)      = 2 - 2(a · b)          ← 내적의 단조감소 함수
```

즉 **정규화만 하면 코사인·내적·L2가 모두 동일한 순위**를 냅니다.
이 노트북은 모든 벡터를 정규화하고, FAISS는 `IndexFlatIP`(내적)를 써서
ChromaDB와 정확히 같은 척도 위에 올려놓습니다.


In [ ]:
# 3️⃣ 성능 측정 도구 — 이후 모든 벤치마크가 이 함수들만 사용합니다
import time
import numpy as np


def measure_latency(search_fn, queries, repeat=20, warmup=2):
    """검색 지연시간 분포를 측정한다 (지표 ②③).

    Args:
        search_fn: 쿼리 벡터 1개를 받아 검색하는 함수
        queries:   쿼리 벡터 배열
        repeat:    전체 쿼리 세트를 몇 번 반복할지
        warmup:    측정에서 제외할 예열 횟수 (첫 호출은 캐시/지연 로딩 때문에 항상 느림)

    Returns:
        {"p50", "p95", "mean", "qps"} — 단위는 ms, QPS는 초당 쿼리 수
    """
    for q in queries[:warmup]:          # 예열: 측정에 포함하지 않는다
        search_fn(q)

    latencies = []
    for _ in range(repeat):
        for q in queries:
            t0 = time.perf_counter()
            search_fn(q)
            latencies.append((time.perf_counter() - t0) * 1000)  # ms

    lat = np.array(latencies)
    return {
        "p50": float(np.percentile(lat, 50)),   # 중앙값
        "p95": float(np.percentile(lat, 95)),   # 20건 중 1건이 겪는 최악
        "mean": float(lat.mean()),
        "qps": float(1000.0 / lat.mean()),
    }


def recall_at_k(ground_truth, predicted, k):
    """완전탐색 정답 대비 근사검색의 Recall@k (지표 ①).

    순서가 아니라 '집합'이 얼마나 겹치는지를 본다.
        정답 [12, 7, 40] / 예측 [12, 40, 55] → 겹침 2개 → 2/3 = 67%
    """
    scores = [len(set(gt[:k]) & set(pred[:k])) / k
              for gt, pred in zip(ground_truth, predicted)]
    return float(np.mean(scores))


def exact_match_rate(ground_truth, predicted, k):
    """순서까지 완전히 같은 쿼리의 비율 (Recall보다 엄격한 보조 지표)."""
    return float(np.mean([list(gt[:k]) == list(pred[:k])
                          for gt, pred in zip(ground_truth, predicted)]))


print("성능 측정 도구 준비 완료")
print("  measure_latency()  → ② 지연(p50/p95), ③ 처리량(QPS)")
print("  recall_at_k()      → ① 검색 품질")
print("  exact_match_rate() → ① 보조: 순위까지 일치하는가")

# 동작 확인: Recall 계산 예시
demo_gt   = [[12, 7, 40, 3, 91]]
demo_pred = [[12, 40, 3, 55, 91]]
print(f"\n[예시] 정답 {demo_gt[0]} vs 근사 {demo_pred[0]}")
print(f"       Recall@5 = {recall_at_k(demo_gt, demo_pred, 5):.0%}")

---

## 4️⃣ 공통 실험 데이터 준비

3원칙의 1번(같은 데이터)을 세팅합니다.
여기서 만든 `documents`, `doc_vectors`, `query_vectors`를 **이후 모든 셀이 공유**합니다.


In [ ]:
# 4️⃣ 공통 실험 데이터 — 3원칙 ①같은 데이터 ②같은 쿼리 ③같은 척도
from sentence_transformers import SentenceTransformer
import numpy as np

# 임베딩 모델 로딩
# 한국어 RAG 최적화 모델 (KLUE-RoBERTa + SimCSE 대조학습)
# 다른 옵션:
#   "sentence-transformers/all-MiniLM-L6-v2"           — 영어 위주, 가벼움 (384d)
#   "paraphrase-multilingual-mpnet-base-v2"            — 다국어 균형 (768d)
#   "BM-K/KoSimCSE-bert-multitask"                     — KoSimCSE BERT 변형
EMBEDDING_MODEL = "BM-K/KoSimCSE-roberta-multitask"
print(f"임베딩 모델 로딩: {EMBEDDING_MODEL}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
dimension = embedding_model.get_sentence_embedding_dimension()
print(f"임베딩 차원: {dimension}")

# --- ① 같은 데이터: 실습용 한국어 샘플 문서 ---
documents = [
    "벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니다. 전통적인 데이터베이스와 달리 유사도 기반 검색을 지원합니다.",
    "FAISS는 Meta에서 개발한 벡터 유사도 검색 라이브러리입니다. GPU 가속을 지원하여 수십억 개의 벡터도 빠르게 검색할 수 있습니다.",
    "ChromaDB는 오픈소스 벡터 데이터베이스로, Python에서 간단한 API로 임베딩을 저장하고 검색할 수 있습니다.",
    "RAG는 Retrieval-Augmented Generation의 약자로, 외부 지식을 검색하여 LLM의 답변 품질을 향상시키는 기술입니다.",
    "트랜스포머 아키텍처의 셀프 어텐션 메커니즘은 입력 시퀀스 내 모든 토큰 간의 관계를 동시에 계산합니다.",
    "Pinecone은 완전 관리형 벡터 데이터베이스 서비스로, 인프라 관리 없이 벡터 검색을 구현할 수 있습니다.",
    "Milvus는 분산 아키텍처를 채택한 벡터 데이터베이스로, 10억 개 이상의 벡터를 처리할 수 있습니다.",
    "임베딩은 텍스트, 이미지 등의 비정형 데이터를 고차원 벡터로 변환하는 과정입니다. 의미가 유사한 데이터는 벡터 공간에서 가까이 위치합니다.",
    "HNSW 알고리즘은 계층적 탐색 가능한 소세계 그래프를 구축하여 근사 최근접 이웃 검색을 수행합니다.",
    "코사인 유사도는 두 벡터 간의 각도를 기반으로 유사도를 측정합니다. 값이 1에 가까울수록 유사하고 0에 가까울수록 다릅니다.",
]

metadatas = [
    {"category": "vector_db", "topic": "개념"},
    {"category": "vector_db", "topic": "FAISS"},
    {"category": "vector_db", "topic": "ChromaDB"},
    {"category": "rag", "topic": "RAG"},
    {"category": "model", "topic": "트랜스포머"},
    {"category": "vector_db", "topic": "Pinecone"},
    {"category": "vector_db", "topic": "Milvus"},
    {"category": "embedding", "topic": "임베딩"},
    {"category": "algorithm", "topic": "HNSW"},
    {"category": "algorithm", "topic": "코사인유사도"},
]

# --- ② 같은 쿼리: 모든 구현체에 동일하게 던질 질의 ---
test_queries = [
    "벡터 검색에 적합한 데이터베이스는?",
    "GPU를 활용한 빠른 벡터 검색",
    "텍스트를 벡터로 변환하는 방법",
    "근사 최근접 이웃 알고리즘의 원리",
    "인프라 관리가 필요 없는 클라우드 검색 서비스",
]

# --- ③ 같은 척도: 정규화하면 코사인 = 내적, L2도 같은 순위 ---
# normalize_embeddings=True → 모든 벡터의 길이(L2 norm)가 1이 된다
doc_vectors = embedding_model.encode(
    documents, normalize_embeddings=True
).astype("float32")
query_vectors = embedding_model.encode(
    test_queries, normalize_embeddings=True
).astype("float32")

print(f"\n문서 {len(documents)}개, 쿼리 {len(test_queries)}개 준비 완료")
print(f"문서 벡터 shape: {doc_vectors.shape}")

# 정규화 검증 — 이게 성립해야 코사인·내적·L2가 같은 순위를 낸다
norms = np.linalg.norm(doc_vectors, axis=1)
print(f"벡터 길이(L2 norm): 최소 {norms.min():.6f} / 최대 {norms.max():.6f}  → 모두 1")

# 실제로 코사인 = 내적임을 확인
a, b = doc_vectors[0], doc_vectors[1]
cosine = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
inner = np.dot(a, b)
l2_sq = np.sum((a - b) ** 2)
print(f"\n문서0 vs 문서1")
print(f"  코사인 유사도      = {cosine:.6f}")
print(f"  내적               = {inner:.6f}   (코사인과 동일)")
print(f"  L2거리² = 2-2·내적 = {l2_sq:.6f} vs {2 - 2 * inner:.6f}  (일치)")

---

## 5️⃣ 구현체 A: ChromaDB

먼저 각 구현체를 **최소한의 코드로** 돌려봅니다. 비교는 6️⃣에서 한꺼번에 합니다.

ChromaDB는 임베디드 벡터 DB입니다. 서버를 띄우지 않고 `chromadb.Client()` 한 줄로 시작하며,
문서 원문·메타데이터·벡터를 **한 곳에 함께 저장**합니다.


In [ ]:
# 5️⃣-A ChromaDB: 적재와 검색
import chromadb

# 인메모리 클라이언트 — 서버 설치 불필요
chroma_client = chromadb.Client()

collection_name = "vector_db_comparison"
try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass

# hnsw:space="cosine" → 거리 척도를 코사인으로 지정 (3원칙 ③)
collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"},
)

# 벡터 + 원문 + 메타데이터를 한 번에 저장하는 것이 ChromaDB의 특징
doc_ids = [f"doc_{i}" for i in range(len(documents))]
collection.add(
    ids=doc_ids,
    embeddings=doc_vectors.tolist(),
    documents=documents,
    metadatas=metadatas,
)
print(f"ChromaDB 컬렉션 '{collection_name}' — 문서 {collection.count()}개 저장\n")

# --- 검색 ---
print("=" * 72)
print("ChromaDB 검색 결과 (상위 3개)")
print("=" * 72)
for query, qvec in zip(test_queries[:3], query_vectors[:3]):
    result = collection.query(
        query_embeddings=[qvec.tolist()],
        n_results=3,
        include=["documents", "metadatas", "distances"],
    )
    print(f"\n[쿼리] {query}")
    for rank, (doc, meta, dist) in enumerate(zip(
        result["documents"][0], result["metadatas"][0], result["distances"][0]
    ), start=1):
        # 코사인 공간에서 Chroma의 distance = 1 - 코사인유사도
        print(f"  {rank}. (유사도 {1 - dist:.4f}) [{meta['topic']}] {doc[:52]}...")

# --- 메타데이터 필터링: ChromaDB의 핵심 강점 (지표 ⑥ 운영 특성) ---
print("\n" + "=" * 72)
print("메타데이터 필터: category='vector_db' 인 문서로 검색 범위 제한")
print("=" * 72)
qvec = query_vectors[2]   # "텍스트를 벡터로 변환하는 방법"
print(f"\n[쿼리] {test_queries[2]}")

for label, where in [("필터 없음", None), ("category=vector_db", {"category": "vector_db"})]:
    kwargs = {"query_embeddings": [qvec.tolist()], "n_results": 3,
              "include": ["documents", "metadatas", "distances"]}
    if where:
        kwargs["where"] = where
    res = collection.query(**kwargs)
    print(f"\n  -- {label} --")
    for rank, (doc, meta, dist) in enumerate(zip(
        res["documents"][0], res["metadatas"][0], res["distances"][0]
    ), start=1):
        print(f"    {rank}. (유사도 {1 - dist:.4f}) [{meta['category']:>10}] {doc[:44]}...")

print("\n→ 필터를 걸면 유사도가 높아도 다른 카테고리 문서는 후보에서 제외된다")

---

## 5️⃣ 구현체 B: FAISS

FAISS는 **DB가 아니라 라이브러리**입니다. 이 차이가 코드에 그대로 드러납니다.

- 저장하는 것은 **벡터뿐** — 문서 원문도 메타데이터도 보관하지 않습니다
- 검색 결과로 돌려주는 것은 **인덱스 번호(정수)** — 원문은 사용자가 직접 매핑해야 합니다
- 메타데이터 필터가 없어 **후처리로 직접 구현**해야 합니다

3원칙의 3번(같은 척도)을 위해 `IndexFlatL2`가 아니라 **`IndexFlatIP`(내적)** 를 씁니다.
4️⃣에서 벡터를 정규화했으므로 내적 = 코사인 유사도이고, ChromaDB와 같은 척도가 됩니다.


In [ ]:
# 5️⃣-B FAISS: 적재와 검색
import faiss

# IndexFlatIP = 내적 기반 완전탐색.
# doc_vectors가 정규화되어 있으므로 내적 = 코사인 유사도 → ChromaDB와 같은 척도
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(doc_vectors)
print(f"FAISS 인덱스에 저장된 벡터 수: {faiss_index.ntotal}")
print(f"FAISS가 저장한 것: 벡터뿐 (원문·메타데이터 없음)\n")

print("=" * 72)
print("FAISS 검색 결과 (상위 3개)")
print("=" * 72)
for query, qvec in zip(test_queries[:3], query_vectors[:3]):
    # 반환값: similarities(내적=코사인), indices(인덱스 번호)
    sims, indices = faiss_index.search(qvec.reshape(1, -1), 3)
    print(f"\n[쿼리] {query}")
    for rank, (sim, idx) in enumerate(zip(sims[0], indices[0]), start=1):
        # 원문은 FAISS가 모른다 — documents[idx]로 직접 되찾아야 한다
        print(f"  {rank}. (유사도 {sim:.4f}) [idx={idx}] {documents[idx][:52]}...")

# --- FAISS에는 메타데이터 필터가 없다: 직접 구현해야 한다 (지표 ⑥) ---
print("\n" + "=" * 72)
print("FAISS 메타데이터 필터 — 내장 기능이 없어 후처리로 직접 구현")
print("=" * 72)
qvec = query_vectors[2]

# 넉넉히 뽑은 뒤 파이썬에서 걸러내는 방식 (over-fetch + post-filter)
OVER_FETCH = len(documents)
sims, indices = faiss_index.search(qvec.reshape(1, -1), OVER_FETCH)

filtered = [(s, i) for s, i in zip(sims[0], indices[0])
            if metadatas[i]["category"] == "vector_db"][:3]

print(f"\n[쿼리] {test_queries[2]} (category=vector_db)")
for rank, (sim, idx) in enumerate(filtered, start=1):
    print(f"  {rank}. (유사도 {sim:.4f}) [{metadatas[idx]['category']:>10}] {documents[idx][:44]}...")

print(f"\n→ 전체 {OVER_FETCH}개를 다 가져와서 걸렀다.")
print("  데이터가 1000만 개이고 조건에 맞는 문서가 희소하면 이 방식은 무너진다.")
print("  이것이 '필터링이 필요하면 FAISS 단독은 부적합'한 이유다.")

---

## 6️⃣ 동일 조건 벤치마크: ChromaDB vs FAISS

드디어 비교입니다. **같은 문서 10개, 같은 쿼리 5개, 같은 코사인 척도**로
3️⃣에서 정의한 지표를 채웁니다.

측정 항목: 적재 시간(④), 검색 지연 p50/p95(②), 처리량 QPS(③), top-3 순위 일치율(①), 운영 특성(⑥)

> 검색 결과가 얼마나 같은지 먼저 보고, 그 다음에 속도를 봅니다. 순서가 중요합니다 —
> **결과가 다르면 속도 비교는 의미가 없기 때문**입니다.


In [ ]:
# 6️⃣ 동일 조건 벤치마크 — 같은 데이터·같은 쿼리·같은 척도
print("=" * 78)
print("실험 조건:  문서 {}개 | 쿼리 {}개 | 척도 코사인 | 임베딩 {}".format(
    len(documents), len(test_queries), EMBEDDING_MODEL))
print("=" * 78)

TOP_K = 3

# ── 적재 시간 측정 (지표 ④) ──────────────────────────────────
try:
    chroma_client.delete_collection("bench_chroma")
except Exception:
    pass

t0 = time.perf_counter()                      # 컬렉션 생성 + 적재만 계측
_c = chroma_client.create_collection(
    name="bench_chroma", metadata={"hnsw:space": "cosine"})
_c.add(ids=doc_ids, embeddings=doc_vectors.tolist(),
       documents=documents, metadatas=metadatas)
chroma_build_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
_f = faiss.IndexFlatIP(dimension)
_f.add(doc_vectors)
faiss_build_ms = (time.perf_counter() - t0) * 1000


# ── 두 구현을 같은 인터페이스로 감싼다: 쿼리 벡터 → 문서 인덱스 리스트 ──
def chroma_search(qvec):
    res = _c.query(query_embeddings=[qvec.tolist()], n_results=TOP_K,
                   include=["distances"])
    return [int(i.split("_")[1]) for i in res["ids"][0]]


def faiss_search(qvec):
    _, indices = _f.search(qvec.reshape(1, -1), TOP_K)
    return indices[0].tolist()


# ── ① 검색 품질: 결과가 같은가? (속도보다 먼저 확인한다) ──────────
print("\n[1] 검색 결과 비교 — 결과가 다르면 속도 비교는 의미가 없다\n")
chroma_ranks = [chroma_search(q) for q in query_vectors]
faiss_ranks = [faiss_search(q) for q in query_vectors]

for query, cr, fr in zip(test_queries, chroma_ranks, faiss_ranks):
    mark = "일치" if cr == fr else "불일치"
    print(f"  [{mark}] {query}")
    print(f"         ChromaDB {cr}   FAISS {fr}")

print(f"\n  top-{TOP_K} 순위 일치율 (Recall)  : {recall_at_k(faiss_ranks, chroma_ranks, TOP_K):.0%}")
print(f"  순서까지 완전 일치한 쿼리 비율 : {exact_match_rate(faiss_ranks, chroma_ranks, TOP_K):.0%}")

# 유사도 값 자체도 같은지 확인
res = _c.query(query_embeddings=[query_vectors[0].tolist()], n_results=TOP_K,
               include=["distances"])
chroma_sims = [1 - d for d in res["distances"][0]]
faiss_sims, _ = _f.search(query_vectors[0].reshape(1, -1), TOP_K)
print(f"\n  유사도 값 비교 (쿼리: {test_queries[0]})")
print(f"    ChromaDB : {[f'{s:.4f}' for s in chroma_sims]}")
print(f"    FAISS    : {[f'{s:.4f}' for s in faiss_sims[0]]}")
print("    → 같은 벡터에 같은 척도를 쓰면 소수점 4자리까지 같은 값이 나온다")

# ── ②③④ 속도 측정 ────────────────────────────────────────
print("\n[2] 속도 — 결과가 같음을 확인했으므로 이제 비교할 수 있다\n")
chroma_lat = measure_latency(chroma_search, query_vectors, repeat=20)
faiss_lat = measure_latency(faiss_search, query_vectors, repeat=20)

print(f"{'항목':<20}{'ChromaDB':>14}{'FAISS':>14}")
print("-" * 52)
print(f"{'④ 적재 시간(ms)':<20}{chroma_build_ms:>14.1f}{faiss_build_ms:>14.2f}")
print(f"{'② 지연 p50(ms)':<20}{chroma_lat['p50']:>14.3f}{faiss_lat['p50']:>14.4f}")
print(f"{'② 지연 p95(ms)':<20}{chroma_lat['p95']:>14.3f}{faiss_lat['p95']:>14.4f}")
print(f"{'③ 처리량(QPS)':<21}{chroma_lat['qps']:>14,.0f}{faiss_lat['qps']:>14,.0f}")

# ── ⑥ 운영 특성: 숫자로 재지 않지만 실무 선택을 좌우한다 ──────────
print("\n[3] 운영 특성 — 숫자로 재지 않지만 선택을 좌우하는 축\n")
ops = [
    ("원문 저장",       "O (documents)", "X (직접 매핑)"),
    ("메타데이터 필터",  "O (where=)",    "X (후처리 구현)"),
    ("영속성",          "O (PersistentClient)", "수동 write_index"),
    ("문서 삭제/수정",   "O (delete/update)", "제한적"),
    ("반환값",          "원문+메타+거리",  "인덱스 번호+거리"),
    ("GPU 가속",        "X",             "O (faiss-gpu)"),
]
print(f"{'항목':<18}{'ChromaDB':<24}{'FAISS':<20}")
print("-" * 62)
for name, c, f in ops:
    print(f"{name:<16}{c:<24}{f:<20}")

print("\n" + "=" * 78)
print("결론: 검색 품질은 동일하다. 갈리는 것은 속도와 운영 특성이다.")
print("  · 검색 결과를 결정하는 것은 DB가 아니라 '임베딩 모델 + 거리 척도'")
print("  · FAISS가 빠른 이유는 알고리즘이 아니라 계층이 얇기 때문 (DB 오버헤드 없음)")
print("  · 문서 10개 규모에서 이 속도 차이는 실무적으로 무의미 — 7️⃣에서 규모를 키운다")
print("=" * 78)

---

## 7️⃣ 인덱스 알고리즘 벤치마크: 규모가 커지면 무엇이 갈리는가

6️⃣에서 문서 10개로는 두 구현의 차이가 거의 없었습니다. **당연합니다 — 10개는 다 뒤져도 순식간**입니다.
차이는 규모에서 나옵니다. 그리고 규모에서 갈리는 것은 **DB 제품이 아니라 인덱스 알고리즘**입니다.

ChromaDB도 Weaviate도 Milvus도 pgvector도 내부적으로 **같은 HNSW/IVF 계열 알고리즘**을 씁니다.
그래서 이 축은 제품 선택과 (거의) 직교합니다.

### 인덱스 유형

| 인덱스 | 원리 | 학습 필요 | 조절 파라미터 |
|--------|------|-----------|---------------|
| `IndexFlatL2` / `IndexFlatIP` | 완전 탐색 — 모든 벡터와 비교 | X | 없음 |
| `IndexIVFFlat` | 벡터를 `nlist`개 클러스터로 나눈 뒤 가까운 클러스터만 탐색 | **O** (k-means) | `nprobe` (탐색할 클러스터 수) |
| `IndexHNSWFlat` | 계층 그래프를 타고 이웃을 따라 내려가며 탐색 | X | `efConstruction`(구축 품질), `efSearch`(탐색 폭) |
| `IndexIVFPQ` | IVF + 벡터를 압축(양자화)해 메모리 절감 | O | `nprobe`, `m`(분할 수) |

### 실험 데이터에 관한 주의

10만 개를 실제 문장으로 임베딩하면 시간이 오래 걸리므로 합성 벡터를 씁니다.
단, **완전 균등난수(`np.random.random`)를 쓰면 안 됩니다.** 고차원 균등난수는
모든 점 사이 거리가 비슷해지는 *차원의 저주* 상태라 ANN이 비정상적으로 나쁘게 나오고,
실제 임베딩의 성질을 전혀 반영하지 못합니다.

실제 문장 임베딩은 **저차원 다양체 위에 주제별로 뭉쳐서** 분포합니다.
아래 `make_embedding_like()`는 그 성질(저차원 구조 + 주제 클러스터)을 모사합니다.


In [ ]:
# 7️⃣ 인덱스 알고리즘 벤치마크 — 10만 벡터
# 소요 시간 약 40초 (HNSW 구축이 대부분)

N_VECTORS = 100_000
DIM = dimension        # 4️⃣의 임베딩 모델과 같은 차원을 쓴다
TOP_K = 10


def make_embedding_like(n, d, intrinsic_dim=48, n_topics=60, spread=0.5, seed=42):
    """실제 문장 임베딩과 비슷한 성질의 합성 벡터를 만든다.

    실제 임베딩의 두 가지 성질을 모사한다:
      1. 저차원 다양체 — 768차원 공간에 있지만 실제 자유도는 훨씬 낮다
      2. 주제 클러스터 — 비슷한 내용끼리 뭉쳐 있다

    np.random.random()으로 만든 균등난수를 쓰면 안 된다. 고차원 균등난수는
    모든 점 사이 거리가 비슷해지는 '차원의 저주' 상태라, ANN 성능이
    비현실적으로 나쁘게 측정된다.
    """
    rng = np.random.default_rng(seed)
    basis = rng.normal(0, 1, (intrinsic_dim, d)).astype("float32")        # 저차원 → 고차원 투영
    topics = rng.normal(0, 1, (n_topics, intrinsic_dim)).astype("float32")  # 주제 중심
    z = (topics[rng.integers(0, n_topics, n)]
         + rng.normal(0, spread, (n, intrinsic_dim)).astype("float32"))
    vectors = (z @ basis).astype("float32")
    faiss.normalize_L2(vectors)      # 3원칙 ③ — 정규화해야 척도가 통일된다
    return vectors


print(f"합성 벡터 생성: {N_VECTORS:,}개 x {DIM}차원 ...")
large_vectors = make_embedding_like(N_VECTORS, DIM)
bench_queries = make_embedding_like(50, DIM, seed=7)   # 학습 데이터에 없는 쿼리
print(f"  데이터 크기: {large_vectors.nbytes / 1e6:.0f}MB\n")

results = []


def bench_index(name, index, needs_training=False, variants=None):
    """인덱스 하나를 구축하고 파라미터를 바꿔가며 5개 지표를 모두 측정한다."""
    t0 = time.perf_counter()
    if needs_training:
        index.train(large_vectors)       # IVF는 k-means 학습이 필요
    index.add(large_vectors)
    build_sec = time.perf_counter() - t0

    # ⑤ 메모리: 직렬화된 인덱스 크기로 근사
    mem_mb = faiss.serialize_index(index).nbytes / 1e6

    rows = []
    for label, apply_param in (variants or [("", lambda: None)]):
        apply_param()
        lat = measure_latency(lambda q: index.search(q.reshape(1, -1), TOP_K),
                              bench_queries, repeat=5)
        _, retrieved = index.search(bench_queries, TOP_K)   # Recall용 일괄 검색
        rows.append({"name": name + label, "build": build_sec, "mem": mem_mb,
                     "p50": lat["p50"], "p95": lat["p95"], "qps": lat["qps"],
                     "indices": retrieved})
    return rows


# [1] 완전탐색 — Recall의 '정답' 역할
print("[1/3] IndexFlatIP (완전탐색) 구축 중 ...")
results += bench_index("Flat (완전탐색)", faiss.IndexFlatIP(DIM))
ground_truth = results[0]["indices"]

# [2] IVF — 클러스터 기반. nprobe로 정확도/속도를 사후 조절
print("[2/3] IndexIVFFlat 구축 중 (k-means 학습 포함) ...")
ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(DIM), DIM, 256, faiss.METRIC_INNER_PRODUCT)
results += bench_index(
    "IVF nprobe=", ivf, needs_training=True,
    variants=[(str(p), (lambda p=p: setattr(ivf, "nprobe", p))) for p in (1, 4, 16, 64)],
)

# [3] HNSW — 그래프 기반. efSearch로 탐색 폭 조절
#     주의: FAISS의 HNSW는 L2에서 가장 안정적이다. 정규화된 벡터에서는
#     L2 순위 == 코사인 순위이므로 METRIC_L2를 그대로 쓴다.
#     efConstruction 기본값 40은 768차원에서 품질이 크게 떨어진다 → 200으로 올린다.
print("[3/3] IndexHNSWFlat 구축 중 (약 20초 소요) ...")
hnsw = faiss.IndexHNSWFlat(DIM, 32)
hnsw.hnsw.efConstruction = 200
results += bench_index(
    "HNSW efSearch=", hnsw,
    variants=[(str(e), (lambda e=e: setattr(hnsw.hnsw, "efSearch", e))) for e in (16, 64, 256)],
)

# ── 결과 표: 3️⃣에서 정의한 5개 축이 그대로 열이 된다 ──────────────
print("\n" + "=" * 88)
print(f"인덱스 알고리즘 비교 ({N_VECTORS:,} 벡터 x {DIM}차원, Recall@{TOP_K})")
print("=" * 88)
print(f"{'인덱스':<22}{'④구축(s)':>10}{'⑤메모리(MB)':>13}{'②p50(ms)':>11}{'②p95(ms)':>11}{'③QPS':>10}{'①Recall':>10}")
print("-" * 88)
for r in results:
    rec = recall_at_k(ground_truth, r["indices"], TOP_K)
    print(f"{r['name']:<20}{r['build']:>10.1f}{r['mem']:>13.0f}"
          f"{r['p50']:>11.2f}{r['p95']:>11.2f}{r['qps']:>10,.0f}{rec:>10.0%}")
print("=" * 88)

# ── 읽는 법 ────────────────────────────────────────────────
flat = results[0]
best_ivf = max((r for r in results if r["name"].startswith("IVF")),
               key=lambda r: recall_at_k(ground_truth, r["indices"], TOP_K))
print(f"""
[표 읽는 법]

1. Flat은 Recall 100%다 — 정의상 그렇다. 이것이 다른 인덱스의 채점 기준이다.
   대신 p50 {flat['p50']:.1f}ms, QPS {flat['qps']:,.0f} 로 가장 느리다.

2. IVF의 nprobe는 '정확도 손잡이'다. 1 → 64로 올리면
   Recall은 올라가고 QPS는 떨어진다. 구축 후에도 언제든 바꿀 수 있다는 점이 강점이다.

3. HNSW는 구축에 {results[-1]['build']:.0f}초가 걸린다 (Flat의 {results[-1]['build'] / max(flat['build'], 1e-9):,.0f}배).
   대신 검색이 빠르고, efSearch로 탐색 폭을 조절한다. 메모리도 그래프만큼 더 쓴다.

4. 결론: '가장 좋은 인덱스'는 없다.
   - 데이터가 작다        → Flat (근사할 이유가 없다)
   - 갱신이 잦다          → IVF (구축이 빠르다)
   - 읽기 위주 + 저지연   → HNSW (구축 비용을 한 번 치르고 계속 회수한다)
""")

---

## 8️⃣ 벡터 DB 선택 가이드: 용도별 추천

### 측정에서 얻은 결론부터

| 질문 | 측정 결과가 말하는 것 |
|------|----------------------|
| 어느 DB가 더 잘 찾나? | **차이 없음.** 같은 임베딩·같은 척도면 검색 결과는 동일 |
| 그럼 무엇이 검색 품질을 결정하나? | **임베딩 모델과 거리 척도.** DB가 아님 |
| DB 선택은 무엇으로 하나? | **운영 특성** — 필터링, 원문 저장, 영속성, 인프라 부담 |
| 규모가 커지면? | **인덱스 알고리즘 선택**이 지배적. 제품 선택과는 별개 축 |

### 의사결정 플로우

```
프로젝트 시작
  │
  ├─ 학습/프로토타입 목적? ──────────── YES ──> ChromaDB
  │
  ├─ 이미 PostgreSQL 사용 중? ────────── YES ──> pgvector
  │
  ├─ 인프라 관리 없이 빠르게 배포? ───── YES ──> Pinecone
  │
  ├─ 최고 검색 속도 필요? (GPU 활용) ── YES ──> FAISS
  │
  ├─ 하이브리드 검색 필요? ──────────── YES ──> Weaviate
  │
  └─ 10억+ 벡터 대규모 시스템? ──────── YES ──> Milvus
```

### 시나리오별 추천

| 시나리오 | 추천 DB | 이유 |
|----------|---------|------|
| 개인 RAG 챗봇 | **ChromaDB** | 간단한 설치, 빠른 프로토타이핑 |
| 사내 문서 검색 시스템 | **Weaviate** / **pgvector** | 하이브리드 검색, 기존 인프라 활용 |
| 대규모 이커머스 추천 | **Milvus** / **FAISS** | 수억 상품 벡터, 고속 검색 |
| 스타트업 MVP | **Pinecone** | 관리 부담 최소, 빠른 출시 |
| 연구/실험 | **FAISS** | 다양한 인덱스 실험, GPU 가속 |
| 기존 PostgreSQL 앱 확장 | **pgvector** | 마이그레이션 불필요, SQL 통합 |

### 규모별 인덱스 선택 (7️⃣ 측정 근거)

| 벡터 수 | 권장 인덱스 | 근거 |
|---------|-------------|------|
| ~1만 | `Flat` (완전탐색) | 어차피 빠름. 근사할 이유가 없음 |
| 1만~100만 | `IVFFlat` | 구축이 빠르고 `nprobe`로 정확도를 사후 조절 가능 |
| 100만~ | `HNSW` | 검색이 가장 빠름. 구축 시간과 메모리는 감수 |
| 메모리 부족 | `IVFPQ` | 압축으로 메모리 대폭 절감, 정확도 일부 손실 |

### 비용 관점 비교

| 구분 | 무료 | 조건부 무료 | 유료 |
|------|------|------------|------|
| **완전 무료** | ChromaDB, FAISS, pgvector | - | - |
| **프리 티어** | - | Pinecone (제한적), Weaviate Cloud | - |
| **자체 호스팅 무료** | - | Milvus, Weaviate | 운영 비용 |
| **관리형 유료** | - | - | Pinecone Pro, Zilliz Cloud |


In [ ]:
# 전체 실습 요약
print("=" * 72)
print("벡터 DB 심층 비교 분석 - 핵심 정리")
print("=" * 72)
print()
print("1. 벡터 DB의 '성능'은 하나의 숫자가 아니라 5개 축이다")
print("   ① Recall@k(품질) ② p50/p95(지연) ③ QPS(처리량)")
print("   ④ build time(색인) ⑤ index size(메모리)  + ⑥ 운영 특성")
print("   - 평균 지연이 아니라 p95를 본다. 평균은 느린 쿼리를 감춘다.")
print()
print("2. 공정한 비교의 3원칙 — 하나라도 어기면 착시다")
print("   같은 데이터 / 같은 쿼리 / 같은 거리 척도")
print("   - 벡터를 정규화하면 코사인 = 내적, L2도 같은 순위가 된다")
print()
print("3. 측정 결과: 같은 조건에서 ChromaDB와 FAISS의 검색 결과는 동일하다")
print("   - 검색 품질을 결정하는 것은 DB가 아니라 임베딩 모델 + 거리 척도")
print("   - DB 선택은 '운영 특성'으로 한다: 필터링, 원문 저장, 영속성, 인프라")
print()
print("4. 규모가 커지면 갈리는 것은 제품이 아니라 인덱스 알고리즘이다")
print("   - Flat  : Recall 100%, 가장 느림          → 소규모")
print("   - IVF   : nprobe로 정확도 조절, 구축 빠름  → 중규모/갱신 잦음")
print("   - HNSW  : 검색 가장 빠름, 구축 느림+메모리 큼 → 대규모/읽기 위주")
print()
print("5. 제품별 요약")
print("   - ChromaDB: 쉬운 API, 필터 내장, 프로토타이핑에 최적")
print("   - FAISS   : 라이브러리(DB 아님), GPU 가속, 필터 직접 구현")
print("   - Pinecone: 완전 관리형, 빠른 배포")
print("   - Weaviate: 하이브리드 검색, GraphQL")
print("   - Milvus  : 초대규모(10억+), 분산 시스템")
print("=" * 72)

---

## 실습 과제

1. **(성능 정의)** 7️⃣ 표에서 `IndexIVFFlat`의 `nprobe`를 1→64로 올릴 때, Recall과 QPS가 각각 몇 배 변하는지 계산하세요.
   "Recall 1%p를 더 얻는 데 QPS를 얼마나 지불했나"를 구간별로 구해보세요.
2. **(p95의 의미)** `measure_latency()`의 `repeat`를 크게 올리고 p50과 p95의 격차를 관찰하세요.
   격차가 벌어지는 원인은 무엇일까요? (힌트: 캐시, GC, 다른 프로세스)
3. **(3원칙 위반 실험)** FAISS를 `IndexFlatIP` 대신 `IndexFlatL2`로 바꾸되 **정규화를 빼고** 6️⃣를 다시 돌려보세요.
   ChromaDB와 순위가 달라지나요? 이때 "FAISS가 더 나쁘다"고 결론지으면 왜 틀린 결론인가요?
4. **(데이터 성질)** 7️⃣의 `make_embedding_like()`를 `np.random.random()`으로 바꿔 돌려보세요.
   Recall이 어떻게 변하며, 그 결과로 인덱스를 고르면 왜 위험한가요?
5. **(직접 확장)** ChromaDB에 자신의 문서 5개를 추가하고, 메타데이터 필터를 건 검색과 걸지 않은 검색의 결과 차이를 확인하세요.

---

## 참고 자료

- [ChromaDB 공식 문서](https://docs.trychroma.com/)
- [FAISS GitHub](https://github.com/facebookresearch/faiss)
- [FAISS 위키 — 인덱스 선택 가이드](https://github.com/facebookresearch/faiss/wiki/Guidelines-to-choose-an-index)
- [ANN-Benchmarks — 알고리즘별 Recall/QPS 공개 벤치마크](https://ann-benchmarks.com/)
- [Pinecone 문서](https://docs.pinecone.io/)
- [Weaviate 문서](https://weaviate.io/developers/weaviate)
- [Milvus 문서](https://milvus.io/docs)
